In [ ]:
# =========================
# 1. Imports
# =========================
import pandas as pd
import numpy as np
import re
import string

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

In [ ]:

nltk.download("stopwords")
nltk.download("wordnet")

In [ ]:
# =========================
# 2. Loading and merging datasets
# =========================
df_fake = pd.read_csv("./datasets/Fake.csv")
df_true = pd.read_csv("./datasets/True.csv")

In [ ]:
df_fake["class"] = 0   # fake
df_true["class"] = 1   # real

In [ ]:
df = pd.concat([df_fake, df_true], axis=0)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:

# only keep relevant columns
df = df.drop(columns=["title", "subject", "date"])

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x="class", data=df)
plt.xticks([0,1], ["Fake", "Real"])
plt.xlabel("News Class")
plt.ylabel("Number of Articles")
plt.title("Class Distribution of Fake and Real News")
plt.show()

In [ ]:
# =========================
# 3. Text Preprocessing
# =========================
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"<.*?>+", "", text)
    text = re.sub(r"[%s]" % re.escape(string.punctuation), "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    return " ".join(tokens)

In [ ]:

df["text"] = df["text"].apply(clean_text)

In [ ]:
X = df["text"]
y = df["class"]

In [ ]:
# =========================
# 4. Train–Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [ ]:
# =========================
# 5. Rule-Based Classifier
# =========================
fake_keywords = [
    "shocking", "breaking", "conspiracy", "fake", "hoax",
    "click here", "you won", "miracle", "secret", "unbelievable"
]

def rule_based_predict(text):
    for word in fake_keywords:
        if word in text:
            return 0  # Fake
    return 1  # Real

y_rule_pred = X_test.apply(rule_based_predict)

print("\nRule-Based Classifier")
print("====================")
print("Accuracy:", accuracy_score(y_test, y_rule_pred))
print(classification_report(y_test, y_rule_pred))

In [ ]:
# =========================
# 5. TF-IDF Vectoriser
# =========================
tfidf = TfidfVectorizer(
    stop_words="english",
    max_df=0.7,
    ngram_range=(1, 2)
)

X_tfidf = tfidf.fit_transform(X)
print("TF-IDF feature matrix shape:", X_tfidf.shape)

In [ ]:
# =========================
# 7. Machine Learning Models
# =========================
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

pipelines = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("tfidf", tfidf),
        ("clf", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    pipelines[name] = pipeline

    print(f"\n{name}")
    print("=" * len(name))
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))
    print(classification_report(y_test, y_pred))

In [ ]:
# =========================
# 8. Confusion Matrix (Example: Logistic Regression)
# =========================
best_model = pipelines["Logistic Regression"]
y_best_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_best_pred)

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix – Logistic Regression")
plt.show()

In [ ]:
# =========================
# 9. Word Cloud Visualisation
# =========================
fake_text = " ".join(df[df["class"] == 0]["clean_text"])
real_text = " ".join(df[df["class"] == 1]["clean_text"])

wc_fake = WordCloud(width=800, height=400, background_color="white").generate(fake_text)
wc_real = WordCloud(width=800, height=400, background_color="white").generate(real_text)

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(wc_fake)
plt.axis("off")
plt.title("Fake News Word Cloud")

In [ ]:

plt.subplot(1, 2, 2)
plt.imshow(wc_real)
plt.axis("off")
plt.title("Real News Word Cloud")
plt.show()

In [ ]:
# =========================
# 10. Manual Prediction Function
# =========================
def predict_news(text):
    text = clean_text(text)
    results = {}

    for name, model in pipelines.items():
        pred = model.predict([text])[0]
        results[name] = "Real News" if pred == 1 else "Fake News"

    return results

In [ ]:
sample_text = "Scientists confirm climate change is accelerating worldwide."
predict_news(sample_text)